# 03 — Giessen External Validation

Loads the Giessen dataset (canonical antibiotic naming applied), builds the split, and (if TensorFlow is available) runs the AMR-EnsembleNet reproduction. See `docs/OPEN_QUESTIONS.md` #7 for why this notebook's split is NOT the same as thesis Table 5.10's.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.amr_data.config import load_paths
from src.amr_data.giessen import load_giessen_raw, build_giessen_split
from src.amr_data import qa

paths = load_paths()
ds = load_giessen_raw(paths)
qa.verify_giessen(ds)

In [ ]:
split = build_giessen_split(ds, seed=42)
print({k: len(v) if hasattr(v, '__len__') else v for k, v in split.items()})

## AMR-EnsembleNet reproduction (requires `tensorflow`)

Attribution: Siddiqui & Tarannum (2025), arXiv:2509.23552, github.com/Saiful185/AMR-EnsembleNet. See `src/amr_data/amr_ensemblenet.py` module docstring -- NOT executed as part of this refactor (no TensorFlow in that environment).

In [ ]:
from src.amr_data import amr_ensemblenet as ae

try:
    import tensorflow  # noqa: F401
    HAS_TF = True
except ModuleNotFoundError:
    HAS_TF = False
    print('TensorFlow not installed -- skipping CNN reproduction. '
          'See docs/OPEN_QUESTIONS.md #10.')

if HAS_TF:
    for ab in ds['label_names']:
        j = ds['antibiotic_to_index'][ab]
        m = ds['label_mask'][:, j]
        tr = split['train_idx'][m[split['train_idx']]]
        va = split['val_idx'][m[split['val_idx']]]
        te = split['test_idx'][m[split['test_idx']]]
        results = ae.run_one_antibiotic(
            ds['X'][tr].astype('int32'), ds['Y'][tr, j].astype('int32'),
            ds['X'][va].astype('int32'), ds['Y'][va, j].astype('int32'),
            ds['X'][te].astype('int32'), ds['Y'][te, j].astype('int32'),
            ab,
        )
        print(ab, results['Ensemble (soft vote, official)'])